In [ ]:
# Colab dependency install (T4 / CUDA + QLoRA)
!pip install -U bitsandbytes transformers peft datasets evaluate bert-score rouge_score huggingface_hub trl accelerate tensorboard

In [ ]:
# Imports
import math
import random
import re
from functools import partial
from typing import Any

import evaluate
import numpy as np
import torch
from datasets import concatenate_datasets, load_dataset
from google.colab import userdata
# from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    pipeline,
)
from tqdm import tqdm


In [ ]:
# Authentication + runtime checks
hf_token = userdata.get("HF_TOKEN_WRITE") # For Google Colab
# hf_token = UserSecretsClient().get_secret("HF_TOKEN") # For Kaggle Notebook
if hf_token:
    login(token=hf_token)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA runtime not detected. Set Colab runtime to GPU (T4).")

print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
SEED = 42  # Global random seed for reproducibility across dataset mapping and training.
MODEL_ID = "google/medgemma-1.5-4b-it"  # Base Hugging Face model ID to fine-tune.
CHECKPOINT_PATH = "gabrielbuzzi/medgemma-aci-lora"  # HF Hub path to fine-tuned LoRA adapter.
PROMPT = """
System Prompt:
You are a medical scribe. Given a transcription of a doctor-patient consultation, generate a structured clinical note in the following format. Write in third person, past tense, using formal medical language. Synthesize and paraphrase — do not copy dialogue verbatim.
Required sections and guidance:
CHIEF COMPLAINT — One sentence stating the reason for the visit.
HISTORY OF PRESENT ILLNESS — Narrative paragraph(s) covering: patient demographics and relevant past medical history (PMH), then chronological account of current status for each active condition, lifestyle factors, medication adherence, and any symptoms endorsed or denied. Group related conditions together.
REVIEW OF SYSTEMS — Bulleted by organ system. Use "Endorses" or "Denies" for each relevant symptom. Only include systems discussed in the encounter.
PHYSICAL EXAMINATION — List findings by system. Only include what was explicitly examined.
VITALS REVIEWED — Summarize notable vitals only (e.g., "Elevated," "Within normal limits"). Omit if not discussed.
RESULTS — Summarize any lab, imaging, or diagnostic results referenced during the visit. Omit if none discussed.
ASSESSMENT AND PLAN — Restate patient demographics and PMH in the opening sentence. Then for each active diagnosis (in order of clinical priority), include relevant subsections chosen from: Medical Reasoning, Additional Testing, Medical Treatment, Patient Education and Counseling. Close with "Patient Agreements" confirming the patient's understanding.
Rules:

Only document what is present in the transcription; do not invent clinical details.
Use standard medical abbreviations and terminology (e.g., dyspnea, ideation, ejection fraction).
If a section has no relevant content from the transcript, omit it entirely.
Keep the tone objective and professional throughout.

Input: A raw transcription of a doctor-patient conversation.
Output: A complete clinical note following the format above.

Input:
{}

Output:
"""  # Single fixed instruction prompt used for all training examples.

TEST_NOISE_PROB = 0.0  # Noise level for test-time prompts during final evaluation.

MAX_NEW_TOKENS = 2048  # Maximum tokens generated per inference sample during evaluation.



TEST_SUBSET_SIZE = 256  # Test subset size for final metric reporting in Colab-friendly runtime.



random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Dataset load
dataset = load_dataset("mkieffer/ACI-Bench", "aci")
print(dataset)


In [ ]:
# Preprocessing helpers

def clean_dialogue(text: str) -> str:
    """Remove speaker tags/newlines to better match raw transcription style."""
    text = re.sub(r"\[doctor\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[patient\]", "", text, flags=re.IGNORECASE)
    text = text.replace("\\n", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def simulate_transcription_noise(text: str, drop_prob: float) -> str:
    """Drop random words to approximate ASR omissions."""
    words = text.split()
    words = [w for w in words if random.random() > drop_prob]
    return " ".join(words)



def format_test_example(prompt: str, example: dict[str, Any], noise_prob: float) -> dict[str, Any]:
    dialogue = clean_dialogue(example["dialogue"])
    if noise_prob > 0:
        dialogue = simulate_transcription_noise(dialogue, drop_prob=noise_prob)

    example["messages"] = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt.format(dialogue)}],
        }
    ]
    return example



## Evaluation


In [ ]:
import torch
import math
import numpy as np
import evaluate

from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from peft import PeftModel
from datasets import concatenate_datasets
from functools import partial
from tqdm import tqdm


BASE_MODEL_ID = "google/medgemma-1.5-4b-it"
CHECKPOINT_PATH = "gabrielbuzzi/medgemma-aci-lora"

MAX_NEW_TOKENS = 1024


print("Loading processor...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"


print("Loading BASE model (no quantization)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

base_model.eval()


print("Loading LoRA on top of BASE...")
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT_PATH,
)

print("Merging LoRA weights...")
fine_tuned_model = fine_tuned_model.merge_and_unload()
fine_tuned_model.eval()


print("Building inference pipelines...")

base_pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
)

ft_pipe = pipeline(
    "text-generation",
    model=fine_tuned_model,
    tokenizer=tokenizer,
)


for pipe in [base_pipe, ft_pipe]:
    pipe.model.generation_config.do_sample = False
    pipe.model.generation_config.temperature = 0.0
    pipe.model.generation_config.pad_token_id = tokenizer.eos_token_id



In [ ]:
from tqdm import tqdm

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def generate_predictions(eval_dataset, pipe, processor, max_new_tokens):
    predictions = []

    for example in tqdm(eval_dataset):
        inference_messages = [
            m for m in example["messages"] if m["role"] != "assistant"
        ]

        prompt_text = processor.tokenizer.apply_chat_template(
            inference_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        output = pipe(
            prompt_text,
            max_new_tokens=max_new_tokens,
            return_full_text=False,
        )

        predictions.append(output[0]["generated_text"].strip())

    return predictions


def compute_cross_entropy_and_perplexity(references, model, tokenizer):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for ref in references:
            inputs = tokenizer(
                ref,
                return_tensors="pt",
                truncation=True,
                padding=True,
            ).to(model.device)

            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

            token_count = int(inputs["input_ids"].numel())
            total_loss += float(loss.item() * token_count)
            total_tokens += token_count

    avg_ce = total_loss / max(total_tokens, 1)
    ppl = float(math.exp(avg_ce)) if avg_ce < 20 else float("inf")

    return avg_ce, ppl



def compute_text_metrics(predictions, references, model, tokenizer):
    print("Computing ROUGE...")
    rouge = rouge_metric.compute(predictions=predictions, references=references)

    print("Computing BLEU...")
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )

    print("Computing BERTScore...")
    bertscore = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="en",
    )

    print("Computing Cross-Entropy + Perplexity...")
    ce, ppl = compute_cross_entropy_and_perplexity(
        references,
        model,
        tokenizer,
    )

    return {
        "rouge1": float(rouge["rouge1"]),
        "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]),
        "bleu": float(bleu["bleu"]),
        "bertscore_f1": float(np.mean(bertscore["f1"])),
        "cross_entropy": ce,
        "perplexity": ppl,
    }

In [ ]:
test_ds = concatenate_datasets([
    dataset["test1"]
])

test_ds = test_ds.map(
    partial(format_test_example, PROMPT, noise_prob=TEST_NOISE_PROB)
)

if TEST_SUBSET_SIZE > 0:
    test_ds = test_ds.select(range(min(TEST_SUBSET_SIZE, len(test_ds))))

references = test_ds["note"]


In [ ]:
print("
Generating FINE-TUNED predictions...")
ft_predictions = generate_predictions(test_ds, ft_pipe, processor, MAX_NEW_TOKENS)

print("
Generating BASE predictions...")
base_predictions = generate_predictions(test_ds, base_pipe, processor, MAX_NEW_TOKENS)


In [ ]:
print("\nGenerating BASE predictions...")
base_predictions = generate_predictions(test_ds, base_pipe, processor, MAX_NEW_TOKENS)

print("\nComputing BASE metrics...")
base_metrics = compute_text_metrics(base_predictions, references, base_model, tokenizer)

print("\nComputing FINE-TUNED metrics...")
ft_metrics = compute_text_metrics(ft_predictions, references, fine_tuned_model, tokenizer)

print("\n================= FINAL COMPARISON =================")

for key in base_metrics.keys():
    print(f"{key}")
    print(f"   Base Model      : {base_metrics[key]:.4f}")
    print(f"   Fine-Tuned Model: {ft_metrics[key]:.4f}")
    print("--------------------------------------------------")